In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import torch
import kagglehub
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
X_train = torch.tensor(X_train.values, dtype=torch.float32)
X_test  = torch.tensor(X_test.values, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.long)
y_test  = torch.tensor(y_test.values, dtype=torch.long)

In [ ]:
# 2. Create TensorDataset objects

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)




In [ ]:
# 3. Create DataLoaders

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)



In [ ]:
# 4. Print shape of one batch

X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")


In [ ]:
# 5. Display sample images

print("Train dataset:", len(train_dataset))
print("Test dataset:", len(test_dataset))

In [ ]:
# Task 1: Write your model class here:
class NN4Layer(nn.Module):
  def __init__(self, input_dim, hidden_dim):
    super(NN3Layer, self).__init__()
    self.layer1 = nn.Linear(input_dim, hidden_dim)

    self.layer2 = nn.Linear(hidden_dim, hidden_dim)
    self.layer3 = nn.Linear(hidden_dim, hidden_dim)

    self.layer4 = nn.Linear(hidden_dim, output_dim)
    # ReLU activation
    self.relu = nn.ReLU()

  def forward(self, x):
    #First hidden layer with ReLU
    a1 = self.relu(self.layer1(x))

    a2 = self.relu(self.layer1(a1))
    a3 = self.relu(self.layer1(a2))
    output = self.layer4(a3)

    return output

In [ ]:
# Task 2: Write your training loop here:
def train_one_epoch(model, optimizer, criterion, train_loader, device):
  # : Set the model to training mode
  model.train()
  running_loss = 0.0

  for X_batch, y_batch in train_loader:
    # : Move batch to the selected device
    X_batch = X_batch.to(device)
    y_batch = y_batch.to(device)
    y_batch = y_batch.view(-1,1)

    # : Forward pass - get model predictions
    outputs = model(X_batch)

    #  Compute loss using criterion
    loss = criterion(outputs, y_batch)

    # Step 1: Clear previous gradients
    optimizer.zero_grad()

    # Step 2: Compute gradients (backward pass)
    loss.backward()
    # Step 3: Update model parameters
    optimizer.step()

    running_loss += loss.item()

  # Calculate average loss over all batches
  avg_loss = running_loss / len(train_loader)

  return avg_loss

In [ ]:
# Task 3: Write your validation loop here:
def validate(model, criterion, test_loader, device):
  model.eval()

  running_loss = 0.0

  with torch.no_grad():
    for X_batch, y_batch in test_loader:
      X_batch = X_batch.to(device)
      y_batch = y_batch.to(device)

      outputs = model(X_batch)
      loss = criterion(outputs, y_batch)

      running_loss += loss.item()

  avg_loss = running_loss / len(test_loader)

  return avg_loss

In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model parameters
input_dim = X_train.shape[1]   # Number of tabular features
hidden_dim = 64                # Design choice

model = NN4Layer(input_dim, hidden_dim).to(device)

print("Model Architecture:\n")
print(model)

# Calculate the total number of trainable parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params}")

In [ ]:
# Task 5: Start training for 20 epochs:
num_epochs = 20
learning_rate = 0.001
train_losses = []
val_losses = []

print('Starting Training...')
for epoch in range(num_epochs):
  train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)  # <Replace None with your code>

  val_loss = validate(model, criterion, test_loader, device)  # <Replace None with your code>

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}')

print('Training Complete!')



In [ ]:
# Get predictions on test set
model.eval()
with torch.no_grad():
  predictions = model(X_test).to(device='cuda')
mse = mean_squared_error(y_train,y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_train,y_pred)

print("Model Evaluation")
print(f"  MAE:  ${mae:.2f}")
print(f"  MSE:  {mse:.2f}")
print(f"  RMSE: ${rmse:.2f}")
print(f"  R2:   {r2:.4f}")

In [ ]:
# Task 1: Write your code here:
# Plotting results
plt.figure(figsize=(7, 5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss (MSE)')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: